# Pedestrian Infrastructure Gap Analysis at MARTA Bus Stops

**Author:** Aidan Moran  
**Date:** May 2026  
**Data Sources:** [MARTA GTFS](https://itsmarta.com/app-developer-resources.aspx) · [OpenStreetMap](https://www.openstreetmap.org/) · [GDOT Crash Data](https://gdot.aashtowaresafety.net/crash-data-dashboard) · [Census ACS](https://api.census.gov)

---

Many MARTA bus stops sit on roads with no sidewalks, forcing riders to walk in traffic or 
cross multiple lanes without crosswalks. This analysis quantifies the problem by combining 
transit stop locations with sidewalk network data, pedestrian crash records, and demographic 
data to answer: **How many MARTA bus stops lack safe pedestrian access, and who is most affected?**

**Study area:** DeKalb County, Georgia — focusing on corridors including N Decatur Rd, 
Clairmont Rd, and E Ponce de Leon Ave between Decatur and Avondale Estates.

## 1. Setup

Import libraries and configure the analysis environment.

In [ ]:
import os
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import folium
from folium.plugins import MarkerCluster
from shapely.geometry import Point, LineString, box
from shapely.ops import unary_union
from dotenv import load_dotenv

# ── Load API keys ──
load_dotenv('../config.env')
CENSUS_API_KEY = os.getenv('CENSUS_API_KEY')

# ── Plot styling ──
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})
sns.set_style('whitegrid')

# ── Color palette ──
MARTA_GOLD   = '#D4A843'
MARTA_BLUE   = '#003DA5'
MARTA_RED    = '#CE1141'
MARTA_GREEN  = '#009A44'
SAFE_GREEN   = '#2ecc71'
WARN_YELLOW  = '#f39c12'
DANGER_RED   = '#e74c3c'

# ── Study area bounding box ──
BBOX_SOUTH, BBOX_WEST = 33.72, -84.35
BBOX_NORTH, BBOX_EAST = 33.82, -84.24

print('Libraries loaded successfully.')
print(f'Census API key: {"configured" if CENSUS_API_KEY else "MISSING — check config.env"}'
      f'\nStudy area: {BBOX_SOUTH}°N–{BBOX_NORTH}°N, {abs(BBOX_WEST)}°W–{abs(BBOX_EAST)}°W')

## 2. Data Acquisition

We combine four data sources to build a complete picture of pedestrian conditions at transit stops:

1. **MARTA GTFS** — bus stop locations from the static transit feed
2. **OpenStreetMap** — sidewalk and cycleway geometries via the Overpass API
3. **GDOT Crash Data** — pedestrian and bicycle crash records for DeKalb County (2021–2025)
4. **Census ACS** — tract-level demographics for equity analysis

### 2.1 MARTA Bus Stop Locations

We load stop locations from the MARTA GTFS feed and filter to our study area. 
The `wheelchair_boarding` field indicates whether a stop has accessibility information — 
the high proportion of stops with no data (value 0) is itself a finding.

In [ ]:
# ── Load MARTA GTFS stops ──
stops_raw = pd.read_csv('../data/raw/marta_gtfs/stops.txt')
print(f'Total MARTA stops: {len(stops_raw):,}')

# Filter to study area bounding box
stops = stops_raw[
    (stops_raw['stop_lat'] >= BBOX_SOUTH) & (stops_raw['stop_lat'] <= BBOX_NORTH) &
    (stops_raw['stop_lon'] >= BBOX_WEST) & (stops_raw['stop_lon'] <= BBOX_EAST)
].copy()

# Exclude rail stations (location_type == 1) — we want bus stops
stops = stops[stops['location_type'] != 1].copy()
print(f'Bus stops in study area: {len(stops):,}')

# Convert to GeoDataFrame
stops_gdf = gpd.GeoDataFrame(
    stops, 
    geometry=gpd.points_from_xy(stops['stop_lon'], stops['stop_lat']),
    crs='EPSG:4326'
)

# Wheelchair boarding summary
wb_counts = stops['wheelchair_boarding'].value_counts()
print(f'\nWheelchair boarding data:')
for val, count in wb_counts.items():
    label = {0: 'No info', 1: 'Accessible', 2: 'Not accessible'}.get(val, f'Unknown ({val})')
    print(f'  {label}: {count} ({count/len(stops)*100:.1f}%)')

stops_gdf.head()

### 2.2 Sidewalk & Cycleway Network (OpenStreetMap)

We query the Overpass API for three types of pedestrian infrastructure:

- **Separate sidewalks** — mapped as independent footway geometries (`highway=footway, footway=sidewalk`)
- **Road sidewalk tags** — roads tagged with `sidewalk=both|left|right|yes`
- **Cycleways** — dedicated bicycle infrastructure (`highway=cycleway`)
- **Crossings** — marked pedestrian crossings (`highway=footway, footway=crossing`)

> **Note on OSM completeness:** Sidewalk mapping is known to be incomplete in most US cities. 
> Gaps in OSM data often — but not always — correspond to gaps in real-world infrastructure. 
> We validate against known conditions on key corridors.

In [ ]:
def query_overpass(query, retries=3):
    """Query the Overpass API with GET request and endpoint fallback."""
    endpoints = [
        'https://overpass-api.de/api/interpreter',
        'https://overpass.kumi.systems/api/interpreter',
    ]
    headers = {
        'User-Agent': 'MARTA-PedestrianAnalysis/1.0 (academic research)',
        'Accept': 'application/json',
    }
    
    last_error = None
    for url in endpoints:
        for attempt in range(retries):
            try:
                print(f'  Trying {url.split("//")[1].split("/")[0]}... (attempt {attempt + 1})')
                resp = requests.get(url, params={'data': query}, 
                                    headers=headers, timeout=120)
                resp.raise_for_status()
                return resp.json()
            except requests.exceptions.RequestException as e:
                last_error = e
                status = getattr(getattr(e, 'response', None), 'status_code', None)
                if status == 429 or status == 504:
                    import time
                    wait = 10 * (attempt + 1)
                    print(f'    Status {status}, waiting {wait}s...')
                    time.sleep(wait)
                elif status == 406:
                    print(f'    Status 406 from this endpoint, trying next...')
                    break
                elif attempt < retries - 1:
                    import time
                    time.sleep(5 * (attempt + 1))
                else:
                    resp_text = getattr(getattr(e, 'response', None), 'text', 'No response body')
                    print(f'    Error details: {resp_text[:500]}')
    
    raise RuntimeError(f'All Overpass API endpoints failed. Last error: {last_error}')

bbox = f'{BBOX_SOUTH},{BBOX_WEST},{BBOX_NORTH},{BBOX_EAST}'

# Query all pedestrian infrastructure types
overpass_query = f'[out:json][timeout:120];(way["highway"="footway"]["footway"="sidewalk"]({bbox});way["highway"="footway"]["footway"="crossing"]({bbox});way["highway"="cycleway"]({bbox});way["highway"]["sidewalk"~"both|left|right|yes"]({bbox}););out body;>;out skel qt;'

print('Querying Overpass API for pedestrian infrastructure...')
osm_data = query_overpass(overpass_query)

# Separate ways and nodes
osm_ways = [e for e in osm_data['elements'] if e['type'] == 'way']
osm_nodes = {e['id']: (e['lon'], e['lat']) for e in osm_data['elements'] if e['type'] == 'node'}

print(f'\nWays returned: {len(osm_ways):,}')
print(f'Nodes returned: {len(osm_nodes):,}')

# Categorize ways
categories = {'sidewalk': 0, 'crossing': 0, 'cycleway': 0, 'road_with_sidewalk': 0}
for way in osm_ways:
    tags = way.get('tags', {})
    hw = tags.get('highway', '')
    fw = tags.get('footway', '')
    sw = tags.get('sidewalk', '')
    
    if hw == 'footway' and fw == 'sidewalk':
        categories['sidewalk'] += 1
    elif hw == 'footway' and fw == 'crossing':
        categories['crossing'] += 1
    elif hw == 'cycleway':
        categories['cycleway'] += 1
    elif sw in ('both', 'left', 'right', 'yes'):
        categories['road_with_sidewalk'] += 1

print(f'\nInfrastructure breakdown:')
for cat, count in categories.items():
    print(f'  {cat}: {count}')

In [ ]:
# ── Convert OSM ways to GeoDataFrame ──
geometries = []
attributes = []

for way in osm_ways:
    tags = way.get('tags', {})
    node_ids = way.get('nodes', [])
    
    # Build LineString from node coordinates
    coords = [osm_nodes[nid] for nid in node_ids if nid in osm_nodes]
    if len(coords) < 2:
        continue
    
    # Classify infrastructure type
    hw = tags.get('highway', '')
    fw = tags.get('footway', '')
    sw = tags.get('sidewalk', '')
    
    if hw == 'footway' and fw == 'sidewalk':
        infra_type = 'sidewalk'
    elif hw == 'footway' and fw == 'crossing':
        infra_type = 'crossing'
    elif hw == 'cycleway':
        infra_type = 'cycleway'
    elif sw in ('both', 'left', 'right', 'yes'):
        infra_type = 'road_with_sidewalk'
    else:
        infra_type = 'other'
    
    geometries.append(LineString(coords))
    attributes.append({
        'osm_id': way['id'],
        'infra_type': infra_type,
        'name': tags.get('name', ''),
        'surface': tags.get('surface', ''),
        'highway': hw,
        'sidewalk_tag': sw,
    })

sidewalks_gdf = gpd.GeoDataFrame(attributes, geometry=geometries, crs='EPSG:4326')

# Project to meters for distance calculations (UTM zone 16N for Atlanta)
sidewalks_utm = sidewalks_gdf.to_crs('EPSG:32616')
stops_utm = stops_gdf.to_crs('EPSG:32616')

print(f'Sidewalk/cycleway segments built: {len(sidewalks_gdf):,}')
print(f'\nBy infrastructure type:')
print(sidewalks_gdf['infra_type'].value_counts().to_string())

# Calculate total length by type
sidewalks_utm_typed = sidewalks_utm.copy()
sidewalks_utm_typed['length_m'] = sidewalks_utm_typed.geometry.length
print(f'\nTotal infrastructure length by type:')
for itype, group in sidewalks_utm_typed.groupby('infra_type'):
    total_km = group['length_m'].sum() / 1000
    print(f'  {itype}: {total_km:.1f} km')

### 2.3 GDOT Crash Data

We load the full DeKalb County crash dataset (2021–2025) and filter to pedestrian and 
bicycle-involved crashes. Crash types are identified by the "First Harmful Event" and 
"Most Harmful Event" fields, which contain `"Pedestrian"` or `"Pedalcyclist"` values 
when vulnerable road users are involved.

In [ ]:
# ── Load GDOT crash data ──
crashes_raw = pd.read_csv('../data/raw/Collisions Dataset.csv')
print(f'Total DeKalb County crashes (2021-2025): {len(crashes_raw):,}')

# Identify pedestrian and bicycle crashes
def is_ped_bike(row):
    """Check if a crash involved a pedestrian or cyclist."""
    text_cols = ['First Harmful Event (Unit Order)', 'Most Harmful Event (Crash Level)',
                 'Operator/Pedestrian Contrib Factor (excl None, Other, No Contrib Factors)',
                 'Manner of Collision (Crash Level) ']
    combined = ' '.join(str(row.get(col, '')) for col in text_cols).lower()
    
    if 'pedestrian' in combined:
        return 'Pedestrian'
    elif 'pedalcycl' in combined or 'bicycle' in combined or 'bike' in combined:
        return 'Bicycle'
    return None

crashes_raw['crash_mode'] = crashes_raw.apply(is_ped_bike, axis=1)
ped_bike_crashes = crashes_raw[crashes_raw['crash_mode'].notna()].copy()

# Clean coordinates
ped_bike_crashes = ped_bike_crashes.dropna(subset=['Latitude', 'Longitude'])
ped_bike_crashes = ped_bike_crashes[
    (ped_bike_crashes['Latitude'] != 0) & (ped_bike_crashes['Longitude'] != 0)
].copy()

print(f'\nPedestrian/bicycle crashes with valid coordinates: {len(ped_bike_crashes):,}')
print(ped_bike_crashes['crash_mode'].value_counts().to_string())

# Filter to study area
crashes_study = ped_bike_crashes[
    (ped_bike_crashes['Latitude'] >= BBOX_SOUTH) & 
    (ped_bike_crashes['Latitude'] <= BBOX_NORTH) &
    (ped_bike_crashes['Longitude'] >= BBOX_WEST) & 
    (ped_bike_crashes['Longitude'] <= BBOX_EAST)
].copy()

print(f'\nIn study area: {len(crashes_study):,}')
print(crashes_study['crash_mode'].value_counts().to_string())

# Severity breakdown
print(f'\nSeverity breakdown (study area):')
print(crashes_study['KABCO Severity'].value_counts().to_string())

# Convert to GeoDataFrame
crashes_gdf = gpd.GeoDataFrame(
    crashes_study,
    geometry=gpd.points_from_xy(crashes_study['Longitude'], crashes_study['Latitude']),
    crs='EPSG:4326'
)
crashes_utm = crashes_gdf.to_crs('EPSG:32616')

### 2.4 Census ACS Demographics

We pull tract-level demographic data from the Census Bureau's ACS 5-year estimates 
for DeKalb County. Key variables include vehicle ownership, transit commute mode share, 
median household income, and race/ethnicity — all relevant to understanding who depends 
on pedestrian infrastructure to access transit.

In [ ]:
# ── Census ACS API query ──
census_url = 'https://api.census.gov/data/2022/acs/acs5'

variables = ','.join([
    'NAME',
    'B08201_001E', 'B08201_002E',
    'B08301_001E', 'B08301_010E', 'B08301_019E',
    'B19013_001E',
    'B03002_001E', 'B03002_003E', 'B03002_004E', 'B03002_012E',
])

params = {
    'get': variables,
    'for': 'tract:*',
    'in': 'state:13 county:089',
    'key': CENSUS_API_KEY,
}

print('Querying Census ACS API...')
resp = requests.get(census_url, params=params, timeout=30)
resp.raise_for_status()
census_data = resp.json()

census_df = pd.DataFrame(census_data[1:], columns=census_data[0])

numeric_cols = [c for c in census_df.columns if c.startswith('B')]
for col in numeric_cols:
    census_df[col] = pd.to_numeric(census_df[col], errors='coerce')

# Census API returns negative values (e.g., -666666666) as error codes for
# missing/suppressed data. Replace these with NaN before computing percentages.
for col in numeric_cols:
    census_df.loc[census_df[col] < 0, col] = np.nan

census_df['pct_zero_vehicle'] = (census_df['B08201_002E'] / census_df['B08201_001E'] * 100).round(1)
census_df['pct_transit'] = (census_df['B08301_010E'] / census_df['B08301_001E'] * 100).round(1)
census_df['pct_walk'] = (census_df['B08301_019E'] / census_df['B08301_001E'] * 100).round(1)
census_df['pct_black'] = (census_df['B03002_004E'] / census_df['B03002_001E'] * 100).round(1)
census_df['pct_hispanic'] = (census_df['B03002_012E'] / census_df['B03002_001E'] * 100).round(1)
census_df['pct_white'] = (census_df['B03002_003E'] / census_df['B03002_001E'] * 100).round(1)
census_df['median_income'] = census_df['B19013_001E']
census_df['tract_pop'] = census_df['B03002_001E']
census_df['tract_commuters'] = census_df['B08301_001E']
census_df['tract_walkers'] = census_df['B08301_019E'] + census_df['B08301_010E']  # walk + transit commuters
census_df['geoid'] = census_df['state'] + census_df['county'] + census_df['tract']

n_missing_income = census_df['median_income'].isna().sum()
print(f'Census tracts loaded: {len(census_df)}')
print(f'  Tracts with missing/suppressed income data: {n_missing_income}')
print(f'\nDeKalb County averages:')
print(f'  Zero-vehicle households: {census_df["pct_zero_vehicle"].median():.1f}% (median)')
print(f'  Transit commuters: {census_df["pct_transit"].median():.1f}% (median)')
print(f'  Median household income: ${census_df["median_income"].median():,.0f} (median)')


In [ ]:
# ── Load Census tract geometries via Tiger/Line ──
tiger_url = (
    'https://www2.census.gov/geo/tiger/TIGER2022/TRACT/'
    'tl_2022_13_tract.zip'
)

print('Downloading Census tract boundaries for Georgia...')
tracts_gdf = gpd.read_file(tiger_url)

tracts_dekalb = tracts_gdf[tracts_gdf['COUNTYFP'] == '089'].copy()
tracts_dekalb = tracts_dekalb.to_crs('EPSG:4326')
tracts_dekalb = tracts_dekalb.merge(census_df, left_on='GEOID', right_on='geoid', how='left')

study_box = box(BBOX_WEST, BBOX_SOUTH, BBOX_EAST, BBOX_NORTH)
tracts_study = tracts_dekalb[tracts_dekalb.intersects(study_box)].copy()

print(f'DeKalb County tracts: {len(tracts_dekalb)}')
print(f'Tracts intersecting study area: {len(tracts_study)}')

tracts_study_utm = tracts_study.to_crs('EPSG:32616')

## 3. Pedestrian Access Scoring

For each MARTA bus stop, we calculate a **Pedestrian Access Score (PAS)** based on 
the sidewalk infrastructure within a 400-meter walking radius (~5 minute walk). 
The score combines four components:

| Component | Weight | What it measures |
|-----------|--------|------------------|
| Sidewalk presence | 40% | Any sidewalk segment within 50m of the stop |
| Network length | 25% | Total meters of sidewalk within 400m |
| Crossing availability | 20% | Marked crossings within 200m |
| Road sidewalk tags | 15% | Whether the road the stop sits on has sidewalk tags |

Scores range from 0 (no infrastructure) to 100 (well-connected).

In [ ]:
# ── Calculate Pedestrian Access Score for each stop ──
BUFFER_WALK = 400    # meters — walkable catchment area
BUFFER_NEAR = 50     # meters — immediate stop vicinity
BUFFER_CROSSING = 200  # meters — crossing search radius

sidewalks_only = sidewalks_utm[sidewalks_utm['infra_type'] == 'sidewalk']
crossings_only = sidewalks_utm[sidewalks_utm['infra_type'] == 'crossing']
cycleways_only = sidewalks_utm[sidewalks_utm['infra_type'] == 'cycleway']
roads_with_sw = sidewalks_utm[sidewalks_utm['infra_type'] == 'road_with_sidewalk']

sw_sindex = sidewalks_only.sindex
cr_sindex = crossings_only.sindex
rw_sindex = roads_with_sw.sindex

results = []

for idx, stop in stops_utm.iterrows():
    stop_point = stop.geometry
    
    # ── Component 1: Sidewalk proximity (binary for v1, distance-decay for v2) ──
    near_buffer = stop_point.buffer(BUFFER_NEAR)
    near_sw_idx = list(sw_sindex.intersection(near_buffer.bounds))
    near_sidewalks = sidewalks_only.iloc[near_sw_idx]
    has_nearby_sidewalk = any(near_sidewalks.intersects(near_buffer))
    
    near_rw_idx = list(rw_sindex.intersection(near_buffer.bounds))
    near_roads = roads_with_sw.iloc[near_rw_idx]
    has_nearby_road_sw = any(near_roads.intersects(near_buffer))
    
    sidewalk_present = 1.0 if (has_nearby_sidewalk or has_nearby_road_sw) else 0.0
    
    # For v2: find distance to nearest sidewalk/road-with-sidewalk (continuous)
    walk_buffer = stop_point.buffer(BUFFER_WALK)
    walk_sw_idx = list(sw_sindex.intersection(walk_buffer.bounds))
    walk_sidewalks = sidewalks_only.iloc[walk_sw_idx]
    walk_sw_in_range = walk_sidewalks[walk_sidewalks.intersects(walk_buffer)]
    
    walk_rw_idx = list(rw_sindex.intersection(walk_buffer.bounds))
    walk_roads = roads_with_sw.iloc[walk_rw_idx]
    walk_rw_in_range = walk_roads[walk_roads.intersects(walk_buffer)]
    
    # Distance to nearest sidewalk infrastructure (any type)
    all_nearby = pd.concat([walk_sw_in_range, walk_rw_in_range])
    if len(all_nearby) > 0:
        dist_nearest = all_nearby.geometry.distance(stop_point).min()
        # Exponential decay: score=1 at 0m, ~0.37 at 100m, ~0.08 at 250m, ~0 at 400m
        proximity_score = np.exp(-dist_nearest / 100)
    else:
        dist_nearest = np.nan
        proximity_score = 0.0
    
    # ── Component 2: Network coverage (same for both versions) ──
    clipped = walk_sw_in_range
    total_length = clipped.geometry.intersection(walk_buffer).length.sum() if len(clipped) > 0 else 0
    network_score = min(total_length / 2000, 1.0)
    
    # For v2: also count road-with-sidewalk length in network
    rw_length = walk_rw_in_range.geometry.intersection(walk_buffer).length.sum() if len(walk_rw_in_range) > 0 else 0
    total_infra_length = total_length + rw_length
    network_score_v2 = min(total_infra_length / 2000, 1.0)
    
    # ── Component 3: Crossing access ──
    cross_buffer = stop_point.buffer(BUFFER_CROSSING)
    cross_idx = list(cr_sindex.intersection(cross_buffer.bounds))
    near_crossings = crossings_only.iloc[cross_idx]
    crossing_count = sum(near_crossings.intersects(cross_buffer))
    crossing_score = min(crossing_count / 3, 1.0)
    
    # ── Component 4: Road sidewalk tag ──
    road_sw_score = 1.0 if has_nearby_road_sw else 0.0
    # For v2: use coverage ratio (length of road-with-sw / total road length proxy)
    road_sw_coverage = min(rw_length / 800, 1.0)  # 800m is ~half the buffer perimeter
    
    # ── Original PAS (v1) — kept for reference ──
    pas_v1 = (sidewalk_present * 0.40 + network_score * 0.25 + 
              crossing_score * 0.20 + road_sw_score * 0.15) * 100
    
    # ── Improved PAS (v2) — all continuous components ──
    # Weights: proximity 30%, network 35%, crossings 20%, road-sw coverage 15%
    pas_v2 = (proximity_score * 0.30 + network_score_v2 * 0.35 + 
              crossing_score * 0.20 + road_sw_coverage * 0.15) * 100
    
    results.append({
        'stop_id': stop['stop_id'],
        'stop_name': stop['stop_name'],
        'stop_lat': stop['stop_lat'],
        'stop_lon': stop['stop_lon'],
        'has_nearby_sidewalk': has_nearby_sidewalk,
        'has_road_sidewalk_tag': has_nearby_road_sw,
        'sidewalk_length_400m': round(total_length, 1),
        'road_sw_length_400m': round(rw_length, 1),
        'crossing_count_200m': crossing_count,
        'dist_nearest_infra_m': round(dist_nearest, 1) if not np.isnan(dist_nearest) else None,
        'pedestrian_access_score': round(pas_v1, 1),
        'pas_v2': round(pas_v2, 1),
    })

scores_df = pd.DataFrame(results)

def score_category(score):
    if score >= 70: return 'Good'
    elif score >= 40: return 'Moderate'
    elif score >= 15: return 'Poor'
    else: return 'No Infrastructure'

scores_df['access_category'] = scores_df['pedestrian_access_score'].apply(score_category)
scores_df['access_category_v2'] = scores_df['pas_v2'].apply(score_category)

print(f'Pedestrian Access Scores calculated for {len(scores_df)} stops')
print(f'\n{"─"*50}')
print(f'Original PAS (v1) — uses binary sidewalk flags:')
print(scores_df['access_category'].value_counts().to_string())
print(f'  Mean: {scores_df["pedestrian_access_score"].mean():.1f}  Median: {scores_df["pedestrian_access_score"].median():.1f}  Std: {scores_df["pedestrian_access_score"].std():.1f}')
print(f'\n{"─"*50}')
print(f'Improved PAS (v2) — all continuous components:')
print(scores_df['access_category_v2'].value_counts().to_string())
print(f'  Mean: {scores_df["pas_v2"].mean():.1f}  Median: {scores_df["pas_v2"].median():.1f}  Std: {scores_df["pas_v2"].std():.1f}')
print(f'\nCorrelation between v1 and v2: {scores_df["pedestrian_access_score"].corr(scores_df["pas_v2"]):.3f}')


## 4. Crash Proximity Analysis

We measure how pedestrian and bicycle crashes cluster around transit stops. 
For each stop, we count crashes within a 200-meter radius and test whether 
stops with lower Pedestrian Access Scores have higher crash densities.

In [ ]:
# ── Calculate crash proximity for each stop ──
CRASH_RADIUS = 200

scores_gdf = gpd.GeoDataFrame(
    scores_df,
    geometry=gpd.points_from_xy(scores_df['stop_lon'], scores_df['stop_lat']),
    crs='EPSG:4326'
).to_crs('EPSG:32616')

crash_sindex = crashes_utm.sindex

crash_counts = []
for idx, stop in scores_gdf.iterrows():
    buffer = stop.geometry.buffer(CRASH_RADIUS)
    candidates = list(crash_sindex.intersection(buffer.bounds))
    nearby = crashes_utm.iloc[candidates]
    n_crashes = sum(nearby.intersects(buffer))
    
    nearby_in = nearby[nearby.intersects(buffer)]
    n_ped = sum(nearby_in['crash_mode'] == 'Pedestrian') if len(nearby_in) > 0 else 0
    n_bike = sum(nearby_in['crash_mode'] == 'Bicycle') if len(nearby_in) > 0 else 0
    
    crash_counts.append({
        'stop_id': stop['stop_id'],
        'crashes_200m': n_crashes,
        'ped_crashes_200m': n_ped,
        'bike_crashes_200m': n_bike,
    })

crash_df = pd.DataFrame(crash_counts)
scores_df = scores_df.merge(crash_df, on='stop_id')

print(f'Crash proximity analysis complete.')
print(f'\nStops with at least 1 crash within 200m: {(scores_df["crashes_200m"] > 0).sum()} '
      f'({(scores_df["crashes_200m"] > 0).mean()*100:.1f}%)')
print(f'\nCrashes near stops by access category:')
crash_by_cat = scores_df.groupby('access_category')['crashes_200m'].agg(['mean', 'sum', 'count'])
crash_by_cat.columns = ['Mean crashes/stop', 'Total crashes', 'N stops']
print(crash_by_cat.to_string())

In [ ]:
# ── Crash proximity analysis: raw and population-normalized ──

# Note: normalized rates require Census data from cell 20.
# This cell shows raw counts; the normalized version follows after the Census join.

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax1 = axes[0]
colors = scores_df['access_category'].map({
    'Good': SAFE_GREEN, 'Moderate': WARN_YELLOW, 
    'Poor': DANGER_RED, 'No Infrastructure': '#8e44ad'
})
ax1.scatter(scores_df['pedestrian_access_score'], scores_df['crashes_200m'],
           c=colors, alpha=0.5, s=30, edgecolors='white', linewidth=0.3)
ax1.set_xlabel('Pedestrian Access Score')
ax1.set_ylabel('Crashes within 200m (Raw Count)')
ax1.set_title('Access Score vs. Crash Proximity (Raw)')

mask = scores_df['crashes_200m'] > 0
if mask.sum() > 10:
    z = np.polyfit(scores_df.loc[mask, 'pedestrian_access_score'], 
                   scores_df.loc[mask, 'crashes_200m'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(0, 100, 100)
    ax1.plot(x_line, p(x_line), '--', color='gray', alpha=0.7, label='Trend')
    ax1.legend()

ax2 = axes[1]
cat_order = ['No Infrastructure', 'Poor', 'Moderate', 'Good']
cat_colors_list = ['#8e44ad', DANGER_RED, WARN_YELLOW, SAFE_GREEN]
means = scores_df.groupby('access_category')['crashes_200m'].mean().reindex(cat_order)
bars = ax2.bar(range(len(means)), means, color=cat_colors_list)
ax2.set_xticks(range(len(means)))
ax2.set_xticklabels(cat_order, rotation=15)
ax2.set_ylabel('Mean Crashes within 200m')
ax2.set_title('Avg Crash Count by Access Category (Raw)')

for i, (val, cat) in enumerate(zip(means, cat_order)):
    n = len(scores_df[scores_df['access_category'] == cat])
    ax2.text(i, val + 0.1, f'n={n}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../assets/ped_access_vs_crashes.png', dpi=200, bbox_inches='tight')
plt.show()
print('Chart saved to assets/ped_access_vs_crashes.png')


## 5. Equity Analysis

Transit-dependent communities — those with lower vehicle ownership, lower incomes, and 
higher shares of minority residents — rely most heavily on walking to reach bus stops. 
We test whether these communities are also the ones with the worst pedestrian infrastructure.

In [ ]:
# ── Join stops to Census tracts ──
scores_gdf_4326 = gpd.GeoDataFrame(
    scores_df,
    geometry=gpd.points_from_xy(scores_df['stop_lon'], scores_df['stop_lat']),
    crs='EPSG:4326'
)

stops_with_demo = gpd.sjoin(scores_gdf_4326, tracts_study[['GEOID', 'geometry',
    'pct_zero_vehicle', 'pct_transit', 'pct_walk', 'pct_black', 'pct_hispanic',
    'pct_white', 'median_income', 'tract_pop', 'tract_commuters', 'tract_walkers',
    'ALAND']], how='left', predicate='within')

# Compute tract-level density (pop per sq km) — ALAND is in sq meters
stops_with_demo['tract_area_km2'] = stops_with_demo['ALAND'] / 1e6
stops_with_demo['pop_density_km2'] = stops_with_demo['tract_pop'] / stops_with_demo['tract_area_km2']

# Normalized crash rate: crashes per 1000 tract residents
stops_with_demo['crashes_per_1k_pop'] = np.where(
    stops_with_demo['tract_pop'] > 0,
    stops_with_demo['crashes_200m'] / stops_with_demo['tract_pop'] * 1000,
    np.nan
)
# Crashes per 1000 walk+transit commuters (more relevant denominator)
stops_with_demo['crashes_per_1k_active'] = np.where(
    stops_with_demo['tract_walkers'] > 0,
    stops_with_demo['crashes_200m'] / stops_with_demo['tract_walkers'] * 1000,
    np.nan
)

print(f'Stops matched to Census tracts: {stops_with_demo["GEOID"].notna().sum()} / {len(stops_with_demo)}')

demo_vars = {
    'pct_zero_vehicle': 'Zero-Vehicle Households (%)',
    'pct_transit': 'Transit Commuters (%)',
    'median_income': 'Median Household Income ($)',
    'pct_black': 'Black Population (%)',
}

print(f'\nCorrelation: Pedestrian Access Score vs. Demographics')
print(f'{"Variable":<35} {"r":>8} {"Direction":>12}')
print('-' * 58)
for var, label in demo_vars.items():
    valid = stops_with_demo[[var, 'pedestrian_access_score']].dropna()
    if len(valid) > 10:
        r = valid[var].corr(valid['pedestrian_access_score'])
        direction = 'better access' if r > 0 else 'worse access'
        print(f'{label:<35} {r:>8.3f} {direction:>12}')

In [ ]:
# ── Equity visualizations ──
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for ax, (var, label) in zip(axes.flat, demo_vars.items()):
    valid = stops_with_demo[[var, 'pedestrian_access_score', 'access_category']].dropna()
    colors = valid['access_category'].map({
        'Good': SAFE_GREEN, 'Moderate': WARN_YELLOW,
        'Poor': DANGER_RED, 'No Infrastructure': '#8e44ad'
    })
    
    ax.scatter(valid[var], valid['pedestrian_access_score'],
              c=colors, alpha=0.4, s=25, edgecolors='white', linewidth=0.3)
    ax.set_xlabel(label)
    ax.set_ylabel('Pedestrian Access Score')
    
    z = np.polyfit(valid[var], valid['pedestrian_access_score'], 1)
    p = np.poly1d(z)
    x_range = np.linspace(valid[var].min(), valid[var].max(), 100)
    ax.plot(x_range, p(x_range), '--', color='gray', alpha=0.7)
    
    r = valid[var].corr(valid['pedestrian_access_score'])
    ax.set_title(f'{label}\n(r = {r:.3f})')

plt.suptitle('Pedestrian Access Score vs. Community Demographics', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../assets/ped_equity_scatter.png', dpi=200, bbox_inches='tight')
plt.show()
print('Chart saved to assets/ped_equity_scatter.png')

In [ ]:
# ── Population-normalized crash proximity ──
# Now that stops are joined to Census tracts, we can normalize crash counts
# by tract population to control for density.

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

cat_colors = {'Good': SAFE_GREEN, 'Moderate': WARN_YELLOW,
              'Poor': DANGER_RED, 'No Infrastructure': '#8e44ad'}
cat_order = ['No Infrastructure', 'Poor', 'Moderate', 'Good']
cat_colors_list = ['#8e44ad', DANGER_RED, WARN_YELLOW, SAFE_GREEN]

valid = stops_with_demo[stops_with_demo['crashes_per_1k_pop'].notna()].copy()
colors = valid['access_category'].map(cat_colors)

# Top-left: PAS vs crashes per 1k population
ax1 = axes[0, 0]
ax1.scatter(valid['pedestrian_access_score'], valid['crashes_per_1k_pop'],
           c=colors, alpha=0.5, s=30, edgecolors='white', linewidth=0.3)
ax1.set_xlabel('Pedestrian Access Score')
ax1.set_ylabel('Crashes per 1,000 Residents')
ax1.set_title('PAS vs. Crashes (Pop-Normalized)')
mask = valid['crashes_per_1k_pop'] > 0
if mask.sum() > 10:
    z = np.polyfit(valid.loc[mask, 'pedestrian_access_score'],
                   valid.loc[mask, 'crashes_per_1k_pop'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(0, 100, 100)
    ax1.plot(x_line, p(x_line), '--', color='gray', alpha=0.7, label='Trend')
    r = valid.loc[mask, 'pedestrian_access_score'].corr(valid.loc[mask, 'crashes_per_1k_pop'])
    ax1.legend(title=f'r = {r:.3f}')

# Top-right: PAS vs crashes per 1k active commuters (walk + transit)
ax2 = axes[0, 1]
valid_active = valid[valid['crashes_per_1k_active'].notna() & np.isfinite(valid['crashes_per_1k_active'])]
colors_active = valid_active['access_category'].map(cat_colors)
ax2.scatter(valid_active['pedestrian_access_score'], valid_active['crashes_per_1k_active'],
           c=colors_active, alpha=0.5, s=30, edgecolors='white', linewidth=0.3)
ax2.set_xlabel('Pedestrian Access Score')
ax2.set_ylabel('Crashes per 1,000 Walk/Transit Commuters')
ax2.set_title('PAS vs. Crashes (Active-Commuter Normalized)')
mask2 = valid_active['crashes_per_1k_active'] > 0
if mask2.sum() > 10:
    z = np.polyfit(valid_active.loc[mask2, 'pedestrian_access_score'],
                   valid_active.loc[mask2, 'crashes_per_1k_active'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(0, 100, 100)
    ax2.plot(x_line, p(x_line), '--', color='gray', alpha=0.7, label='Trend')
    r = valid_active.loc[mask2, 'pedestrian_access_score'].corr(valid_active.loc[mask2, 'crashes_per_1k_active'])
    ax2.legend(title=f'r = {r:.3f}')

# Bottom-left: bar chart — mean crashes per 1k pop by category
ax3 = axes[1, 0]
means_norm = valid.groupby('access_category')['crashes_per_1k_pop'].mean().reindex(cat_order)
bars = ax3.bar(range(len(means_norm)), means_norm, color=cat_colors_list)
ax3.set_xticks(range(len(means_norm)))
ax3.set_xticklabels(cat_order, rotation=15)
ax3.set_ylabel('Mean Crashes per 1,000 Residents')
ax3.set_title('Avg Crash Rate by Category (Pop-Normalized)')
for i, (val, cat) in enumerate(zip(means_norm, cat_order)):
    n = len(valid[valid['access_category'] == cat])
    ax3.text(i, val + 0.02, f'n={n}', ha='center', fontsize=9)

# Bottom-right: pop density vs PAS to show the confound
ax4 = axes[1, 1]
valid_dens = valid[valid['pop_density_km2'].notna()]
colors_dens = valid_dens['access_category'].map(cat_colors)
ax4.scatter(valid_dens['pop_density_km2'], valid_dens['pedestrian_access_score'],
           c=colors_dens, alpha=0.4, s=25, edgecolors='white', linewidth=0.3)
ax4.set_xlabel('Tract Population Density (per km\u00b2)')
ax4.set_ylabel('Pedestrian Access Score')
ax4.set_title('Population Density vs. PAS')
r_dens = valid_dens['pop_density_km2'].corr(valid_dens['pedestrian_access_score'])
ax4.text(0.95, 0.05, f'r = {r_dens:.3f}', transform=ax4.transAxes,
        ha='right', fontsize=11, style='italic')

plt.suptitle('Crash Proximity Analysis: Raw vs. Population-Normalized', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../assets/ped_crashes_normalized.png', dpi=200, bbox_inches='tight')
plt.show()

print('\nNormalized crash summary by access category:')
summary = valid.groupby('access_category').agg(
    raw_crashes=('crashes_200m', 'mean'),
    per_1k_pop=('crashes_per_1k_pop', 'mean'),
    per_1k_active=('crashes_per_1k_active', lambda x: x[np.isfinite(x)].mean()),
    pop_density=('pop_density_km2', 'mean'),
    n_stops=('crashes_200m', 'count')
).reindex(cat_order)
print(summary.round(2).to_string())
print(f'\nChart saved to assets/ped_crashes_normalized.png')


## 6. Interactive Map

This map shows every MARTA bus stop in the study area, colored by Pedestrian Access Score. 
Crash locations are overlaid as red markers. Click any stop to see its score breakdown.

In [ ]:
# ── Build interactive Folium map ──
center_lat = (BBOX_SOUTH + BBOX_NORTH) / 2
center_lon = (BBOX_WEST + BBOX_EAST) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles='CartoDB positron')

# Add sidewalk network
for _, row in sidewalks_gdf.iterrows():
    color = {'sidewalk': '#2ecc71', 'crossing': '#3498db', 
             'cycleway': '#9b59b6', 'road_with_sidewalk': '#27ae60',
             'other': '#95a5a6'}.get(row['infra_type'], '#95a5a6')
    weight = 2 if row['infra_type'] in ('sidewalk', 'cycleway') else 1
    coords = list(row.geometry.coords)
    folium.PolyLine(
        locations=[(c[1], c[0]) for c in coords],
        color=color, weight=weight, opacity=0.6,
        tooltip=f"{row['infra_type']}: {row['name'] or 'unnamed'}"
    ).add_to(m)

# Add stops colored by PAS
for _, stop in scores_df.iterrows():
    score = stop['pedestrian_access_score']
    cat = stop['access_category']
    color = {'Good': 'green', 'Moderate': 'orange', 
             'Poor': 'red', 'No Infrastructure': 'purple'}.get(cat, 'gray')
    popup_html = (f"<b>{stop['stop_name']}</b><br>"
                  f"<b>Pedestrian Access Score: {score:.0f}/100</b><br>"
                  f"Category: {cat}<br><hr>"
                  f"Nearby sidewalk: {'Yes' if stop['has_nearby_sidewalk'] else 'No'}<br>"
                  f"Road sidewalk tag: {'Yes' if stop['has_road_sidewalk_tag'] else 'No'}<br>"
                  f"Sidewalk within 400m: {stop['sidewalk_length_400m']:.0f}m<br>"
                  f"Crossings within 200m: {stop['crossing_count_200m']}<br>"
                  f"Crashes within 200m: {stop['crashes_200m']} "
                  f"(ped: {stop['ped_crashes_200m']}, bike: {stop['bike_crashes_200m']})")
    folium.CircleMarker(
        location=[stop['stop_lat'], stop['stop_lon']],
        radius=5, color=color, fill=True, fillColor=color, fillOpacity=0.7,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{stop['stop_name']} \u2014 PAS: {score:.0f}"
    ).add_to(m)


# Add crash locations
for _, crash in crashes_gdf.iterrows():
    color = '#e74c3c' if crash['crash_mode'] == 'Pedestrian' else '#3498db'
    label = crash['crash_mode']
    folium.CircleMarker(
        location=[crash.geometry.y, crash.geometry.x],
        radius=4, color=color, fill=True, fillColor=color, fillOpacity=0.9,
        popup=f"{crash['crash_mode']} crash ({crash.get('year', 'N/A')})",
        tooltip=f"{crash['crash_mode']} crash"
    ).add_to(m)

print(f'Added {len(crashes_gdf)} crash markers to map')

# Legend
legend_html = (
    '<div style="position:fixed; bottom:30px; left:30px; z-index:1000; '
    'background:white; padding:12px; border-radius:8px; border:1px solid #ccc; '
    'font-size:12px; max-width:180px;">'
    '<b>Pedestrian Access Score</b><br>'
    '<hr style="margin:4px 0;">'
    '<span style="color:green;">\u25cf</span> Good (70\u2013100) <br>'
    '<span style="color:orange;">\u25cf</span> Moderate (40\u201369) <br>'
    '<span style="color:red;">\u25cf</span> Poor (15\u201339) <br>'
    '<span style="color:purple;">\u25cf</span> No Infrastructure (0\u201314) <br>'
    '<hr style="margin:4px 0;">'
    '<span style="color:green;">\u2500\u2500</span> Sidewalk &nbsp;'
    '<span style="color:#9b59b6;">\u2500\u2500</span> Cycleway<br>'
    '<hr style="margin:4px 0;">'
    '<b>Crashes (2021\u20132025)</b><br>'
    '<span style="color:#e74c3c;">\u25cf</span> Pedestrian<br>'
    '<span style="color:#3498db;">\u25cf</span> Bicycle'
    '</div>'
)

m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl().add_to(m)

m.save('../maps/dekalb_pedestrian_access.html')
print(f'Map saved to maps/dekalb_pedestrian_access.html')
m

## 7. Score Distributions & Corridor Analysis

In [ ]:
# ── Score distribution: v1 vs v2 comparison ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cat_colors = {'No Infrastructure': '#8e44ad', 'Poor': DANGER_RED, 
              'Moderate': WARN_YELLOW, 'Good': SAFE_GREEN}

# Left: v1 histogram
ax1 = axes[0]
for cat in ['No Infrastructure', 'Poor', 'Moderate', 'Good']:
    subset = scores_df[scores_df['access_category'] == cat]
    ax1.hist(subset['pedestrian_access_score'], bins=25, alpha=0.7,
            color=cat_colors[cat], label=f'{cat} ({len(subset)})')
ax1.set_xlabel('Pedestrian Access Score')
ax1.set_ylabel('Number of Stops')
ax1.set_title('PAS v1 (Binary Flags)')
ax1.legend(fontsize=7)

# Center: v2 histogram
ax2 = axes[1]
for cat in ['No Infrastructure', 'Poor', 'Moderate', 'Good']:
    subset = scores_df[scores_df['access_category_v2'] == cat]
    ax2.hist(subset['pas_v2'], bins=25, alpha=0.7,
            color=cat_colors[cat], label=f'{cat} ({len(subset)})')
ax2.set_xlabel('Pedestrian Access Score')
ax2.set_ylabel('Number of Stops')
ax2.set_title('PAS v2 (Continuous Components)')
ax2.legend(fontsize=7)

# Right: v1 vs v2 scatter
ax3 = axes[2]
colors = scores_df['access_category'].map(cat_colors)
ax3.scatter(scores_df['pedestrian_access_score'], scores_df['pas_v2'],
           c=colors, alpha=0.4, s=20, edgecolors='white', linewidth=0.3)
ax3.plot([0, 100], [0, 100], '--', color='gray', alpha=0.5, label='y = x')
ax3.set_xlabel('PAS v1 (Original)')
ax3.set_ylabel('PAS v2 (Improved)')
r = scores_df['pedestrian_access_score'].corr(scores_df['pas_v2'])
ax3.set_title(f'v1 vs v2 (r = {r:.3f})')
ax3.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../assets/ped_score_distribution.png', dpi=200, bbox_inches='tight')
plt.show()
print('Chart saved to assets/ped_score_distribution.png')


In [ ]:
# ── Interactive map: Bus Stops by PAS v2 Score ──
# A clean view of just the stops — no infrastructure lines — for spatial score patterns.

center_lat = (BBOX_SOUTH + BBOX_NORTH) / 2
center_lon = (BBOX_WEST + BBOX_EAST) / 2

m2 = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles='CartoDB positron')

# Deduplicate co-located stop pairs (keep lower score to highlight gaps)
stops_deduped = scores_df.copy()
stops_deduped['loc_key'] = (stops_deduped['stop_lat'].round(4).astype(str) + '_' + 
                            stops_deduped['stop_lon'].round(4).astype(str))
n_before = len(stops_deduped)
stops_deduped = stops_deduped.sort_values('pas_v2').drop_duplicates('loc_key', keep='first')
print(f'Deduplicated: {n_before} \u2192 {len(stops_deduped)} stops '
      f'({n_before - len(stops_deduped)} co-located pairs merged)')

def score_to_color_v2(score):
    """Continuous gradient: purple(0) \u2192 red(15) \u2192 orange(40) \u2192 green(100)"""
    if score < 15:
        return '#8e44ad'
    elif score < 40:
        t = (score - 15) / 25
        g = int(t * 126)
        return f'#e6{g:02x}20'
    elif score < 70:
        t = (score - 40) / 30
        r = int(230 - t * 30)
        g = int(126 + t * 80)
        return f'#{r:02x}{g:02x}20'
    else:
        t = min((score - 70) / 30, 1.0)
        r = int(46 * (1 - t))
        return f'#{r:02x}a832'

for _, stop in stops_deduped.iterrows():
    score_v2 = stop['pas_v2']
    score_v1 = stop['pedestrian_access_score']
    cat_v2 = stop['access_category_v2']
    color = score_to_color_v2(score_v2)
    
    dist_info = f"{stop['dist_nearest_infra_m']:.0f}m" if stop['dist_nearest_infra_m'] is not None else 'None found'
    crashes = int(stop['crashes_200m'])
    crash_flag = f' \u26a0\ufe0f' if crashes > 0 else ''
    
    popup_html = (
        f"<b>{stop['stop_name']}</b>{crash_flag}<br>"
        f"<b>PAS v2: {score_v2:.1f}/100</b> ({cat_v2})<br>"
        f"PAS v1: {score_v1:.1f}/100<br>"
        f"<hr style='margin:4px 0;'>"
        f"<b>Score components:</b><br>"
        f"\u2022 Nearest infra: {dist_info}<br>"
        f"\u2022 Sidewalk network (400m): {stop['sidewalk_length_400m']:.0f}m<br>"
        f"\u2022 Road-with-SW (400m): {stop['road_sw_length_400m']:.0f}m<br>"
        f"\u2022 Crossings (200m): {stop['crossing_count_200m']}<br>"
        f"<hr style='margin:4px 0;'>"
        f"Crashes (200m): {crashes} "
        f"(ped: {int(stop['ped_crashes_200m'])}, bike: {int(stop['bike_crashes_200m'])})")
    
    # Worse stops render larger
    radius = max(4, 10 - score_v2 / 15)
    
    folium.CircleMarker(
        location=[stop['stop_lat'], stop['stop_lon']],
        radius=radius, color=color, fill=True, fillColor=color, fillOpacity=0.8,
        popup=folium.Popup(popup_html, max_width=320),
        tooltip=f"{stop['stop_name']} \u2014 PAS v2: {score_v2:.1f}"
    ).add_to(m2)

# Legend
legend_html = (
    '<div style="position:fixed; bottom:30px; left:30px; z-index:1000; '
    'background:white; padding:12px; border-radius:8px; border:1px solid #ccc; '
    'font-size:12px; max-width:200px;">'
    '<b>PAS v2 Score</b><br>'
    '<hr style="margin:4px 0;">'
    '<span style="color:#00a832;">\u25cf</span> Good (70\u2013100)<br>'
    '<span style="color:#c8ce20;">\u25cf</span> Moderate (40\u201369)<br>'
    '<span style="color:#e67e20;">\u25cf</span> Poor (15\u201339)<br>'
    '<span style="color:#8e44ad;">\u25cf</span> No Infrastructure (0\u201314)<br>'
    '<hr style="margin:4px 0;">'
    '<small>Larger dots = lower scores</small>'
    '</div>'
)

m2.get_root().html.add_child(folium.Element(legend_html))

m2.save('../maps/dekalb_stops_v2_scores.html')
print(f'Map saved to maps/dekalb_stops_v2_scores.html')
m2

In [ ]:
# ── Corridor-level analysis ──
corridors = {
    'N Decatur Rd': scores_df[scores_df['stop_name'].str.contains('N DECATUR|DECATUR RD', case=False, na=False)],
    'Clairmont Rd': scores_df[scores_df['stop_name'].str.contains('CLAIRMONT', case=False, na=False)],
    'E Ponce de Leon': scores_df[scores_df['stop_name'].str.contains('PONCE', case=False, na=False)],
    'Memorial Dr': scores_df[scores_df['stop_name'].str.contains('MEMORIAL', case=False, na=False)],
    'Columbia Dr': scores_df[scores_df['stop_name'].str.contains('COLUMBIA', case=False, na=False)],
}

print(f'{"Corridor":<25} {"Stops":>6} {"Mean PAS":>10} {"No Infra":>10} {"Crashes":>10}')
print('-' * 65)
for name, df in corridors.items():
    if len(df) > 0:
        print(f'{name:<25} {len(df):>6} {df["pedestrian_access_score"].mean():>10.1f} '
              f'{(df["access_category"] == "No Infrastructure").sum():>10} '
              f'{df["crashes_200m"].sum():>10}')

fig, ax = plt.subplots(figsize=(12, 5))
corridor_names = [c for c in corridors if len(corridors[c]) > 0]
corridor_means = [corridors[c]['pedestrian_access_score'].mean() for c in corridor_names]
corridor_colors = [SAFE_GREEN if m >= 70 else WARN_YELLOW if m >= 40 else DANGER_RED for m in corridor_means]

bars = ax.bar(corridor_names, corridor_means, color=corridor_colors, edgecolor='white')
ax.axhline(y=scores_df['pedestrian_access_score'].mean(), color='gray', linestyle='--', 
           alpha=0.7, label=f'Study area average ({scores_df["pedestrian_access_score"].mean():.1f})')
ax.set_ylabel('Mean Pedestrian Access Score')
ax.set_title('Pedestrian Access by Corridor')
ax.legend()

for bar, val in zip(bars, corridor_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.0f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../assets/ped_corridor_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print('Chart saved to assets/ped_corridor_comparison.png')

## 8. Key Findings

In [ ]:
# ── Summary statistics for findings narrative ──
total_stops = len(scores_df)
no_infra = (scores_df['access_category'] == 'No Infrastructure').sum()
poor = (scores_df['access_category'] == 'Poor').sum()
no_sidewalk = (~scores_df['has_nearby_sidewalk'] & ~scores_df['has_road_sidewalk_tag']).sum()
stops_with_crashes = (scores_df['crashes_200m'] > 0).sum()

# v2 score stats
no_infra_v2 = (scores_df['access_category_v2'] == 'No Infrastructure').sum()
poor_v2 = (scores_df['access_category_v2'] == 'Poor').sum()

# Normalized crash stats
valid_norm = stops_with_demo[stops_with_demo['crashes_per_1k_pop'].notna()]
valid_active = stops_with_demo[stops_with_demo['crashes_per_1k_active'].notna() & 
                               np.isfinite(stops_with_demo['crashes_per_1k_active'])]

print('=' * 65)
print('KEY FINDINGS')
print('=' * 65)
print(f'''
1. INFRASTRUCTURE GAPS
   \u2022 {no_infra + poor} of {total_stops} bus stops ({(no_infra+poor)/total_stops*100:.0f}%) have 
     Poor or No pedestrian infrastructure within walking distance (v1)
   \u2022 {no_sidewalk} stops ({no_sidewalk/total_stops*100:.0f}%) have no sidewalk at all 
     within 50 meters
   \u2022 Mean PAS v1: {scores_df['pedestrian_access_score'].mean():.1f}/100
   \u2022 Mean PAS v2 (continuous): {scores_df['pas_v2'].mean():.1f}/100
   \u2022 PAS v2 std dev: {scores_df['pas_v2'].std():.1f} (vs v1: {scores_df['pedestrian_access_score'].std():.1f})

2. SCORING METHODOLOGY
   \u2022 PAS v1 uses binary flags (sidewalk yes/no), creating a bimodal distribution
     with a spike around 80\u201385 and a cluster at 0
   \u2022 PAS v2 replaces binary flags with continuous metrics (distance decay,
     coverage ratios), producing better discrimination between stops
   \u2022 v1\u2013v2 correlation: {scores_df['pedestrian_access_score'].corr(scores_df['pas_v2']):.3f}
''')

print(f'''\
3. CRASH PROXIMITY & DENSITY
   \u2022 {stops_with_crashes} stops ({stops_with_crashes/total_stops*100:.0f}%) had at least one 
     ped/bike crash within 200m (2021\u20132025)
   \u2022 Raw crash counts correlate POSITIVELY with PAS (r = {valid_norm['pedestrian_access_score'].corr(valid_norm['crashes_200m']):.3f})
     \u2014 better infrastructure near more crashes, not fewer
   \u2022 Population-normalized (per 1k residents): r = {valid_norm['pedestrian_access_score'].corr(valid_norm['crashes_per_1k_pop']):.3f}
   \u2022 Active-commuter normalized (per 1k walk/transit): r = {valid_active['pedestrian_access_score'].corr(valid_active['crashes_per_1k_active']):.3f}
   \u2022 Interpretation: the PAS\u2013crash relationship largely disappears after
     normalizing for population. Both crashes and infrastructure concentrate
     along high-traffic corridors. Traffic volume (AADT), not pedestrian
     infrastructure quality, is likely the primary crash predictor.
''')

if 'median_income' in stops_with_demo.columns:
    valid_income = stops_with_demo[stops_with_demo['median_income'].notna()]
    low_income = valid_income[valid_income['median_income'] < valid_income['median_income'].median()]
    high_income = valid_income[valid_income['median_income'] >= valid_income['median_income'].median()]
    r_black = stops_with_demo[['pct_black', 'pedestrian_access_score']].dropna().corr().iloc[0,1]
    
    print(f'''\
4. EQUITY
   \u2022 Strongest demographic correlation: Black population share (r = {r_black:.3f})
     \u2014 areas with higher Black population have lower pedestrian access scores
   \u2022 Stops in lower-income tracts: mean PAS = {low_income['pedestrian_access_score'].mean():.1f}
   \u2022 Stops in higher-income tracts: mean PAS = {high_income['pedestrian_access_score'].mean():.1f}
   \u2022 Income gap: {abs(high_income['pedestrian_access_score'].mean() - low_income['pedestrian_access_score'].mean()):.1f} points
''')

print('''\
5. RECOMMENDATIONS
   \u2022 Priority corridors for sidewalk investment: stops scoring 0 on PAS
     clustered along Valley Brook Rd, Northern Ave, and Flat Shoals Rd SE
   \u2022 Future analysis should incorporate GDOT AADT traffic volume data to
     properly model crash risk vs. infrastructure quality
   \u2022 Ground-truth validation of OSM sidewalk data recommended \u2014 OSM
     coverage may undercount informal paths and unpaved walkways
''')


## 9. Data Export

In [ ]:
# ── Export processed data ──
export_cols = ['stop_id', 'stop_name', 'stop_lat', 'stop_lon',
               'pedestrian_access_score', 'pas_v2', 'access_category', 'access_category_v2',
               'has_nearby_sidewalk', 'has_road_sidewalk_tag',
               'sidewalk_length_400m', 'road_sw_length_400m', 'crossing_count_200m',
               'dist_nearest_infra_m',
               'crashes_200m', 'ped_crashes_200m', 'bike_crashes_200m']

scores_df[export_cols].to_csv('../data/processed/dekalb_stop_ped_access_scores.csv', index=False)
print(f'Exported {len(scores_df)} stop scores to data/processed/dekalb_stop_ped_access_scores.csv')

crashes_study.to_csv('../data/processed/dekalb_ped_bike_crashes.csv', index=False)
print(f'Exported {len(crashes_study)} crashes to data/processed/dekalb_ped_bike_crashes.csv')


---

## What This Analysis Shows

This notebook establishes a quantitative baseline for pedestrian infrastructure at MARTA 
bus stops in DeKalb County. The core finding is stark: **266 of 875 bus stops (30%) have 
Poor or No pedestrian infrastructure** within walking distance, according to OpenStreetMap data.

This isn't random. The equity analysis reveals a pattern of disinvestment: areas with higher 
Black population share have systematically lower Pedestrian Access Scores (r = −0.49). 
MARTA expects riders to walk to these stops, but the surrounding built environment was never 
designed for walking — particularly in neighborhoods built around dendritic, cul-de-sac 
street networks with no pedestrian cut-throughs.

The crash proximity analysis adds nuance: the relationship between infrastructure quality 
and crash rates disappears once you control for population density (active-commuter 
normalized r ≈ 0). This likely reflects **suppressed pedestrian demand** — people simply 
don't walk where there are no sidewalks, so crashes don't occur. The absence of crashes 
at zero-infrastructure stops is not evidence of safety; it's evidence of isolation.

## Limitations

- **OSM coverage gaps:** OpenStreetMap may undercount informal paths, unpaved walkways, 
  and recently constructed sidewalks. Ground-truth validation against satellite imagery 
  or Google Street View is needed for key corridors.
- **Crow-flies buffers vs. network distance:** The 400m walking buffer uses Euclidean 
  distance, not the actual street network. In areas with dendritic street layouts, the 
  real walking distance to a bus stop can be 3–4x the straight-line distance.
- **No traffic volume data:** Without GDOT AADT (Annual Average Daily Traffic) counts, 
  we cannot properly model crash risk. The current analysis cannot separate "dangerous 
  road" from "busy road."
- **No ridership data:** We cannot directly measure suppressed demand without stop-level 
  boarding/alighting counts from MARTA's APC (Automatic Passenger Counter) system.

## Future Work (Notebook 4b)

The following data sources would strengthen this analysis from an exploratory baseline 
into an actionable advocacy tool:

1. **MARTA APC ridership data** — Stop-level boardings would test the suppressed demand 
   hypothesis: do zero-infrastructure stops have lower ridership than comparable stops 
   with sidewalks? If yes, that's a quantifiable cost of missing infrastructure in terms 
   of lost transit trips.

2. **Network-based walk-shed analysis** — Using OSMnx to compute actual walking distances 
   along the street network (not crow-flies buffers) would expose the dendritic problem: 
   stops that are 200m away as the crow flies but 800m+ by foot because of cul-de-sac 
   layouts with no pedestrian connections. These visualizations make the case for 
   cut-through paths and street connectivity improvements.

3. **GDOT AADT traffic volumes** — Adding vehicle traffic counts at nearby road segments 
   would allow proper crash rate modeling (crashes per vehicle-mile or per pedestrian 
   exposure) and identify the most dangerous crossings for pedestrians.

4. **DeKalb County CIP & Atlanta Connect plan** — Cross-referencing identified gaps 
   against planned capital improvements would show which of these 266 underserved stops 
   are already in the pipeline and which are being ignored by current planning.

5. **Before/after case study: N Decatur Rd** — The planned road diet (4-lane to 3-lane 
   conversion with multiuse path) is an opportunity to model how Complete Streets 
   investments change Pedestrian Access Scores and, once implemented, to measure actual 
   ridership and safety impacts.

---

*This analysis is part of a [transit data analytics portfolio](https://aimo1223.github.io/transit-portfolio/). 
Data sources and methodology details are documented in the 
[project scope](../docs/project_scope_pedestrian_gaps.md).*